# Step 0 公式実行ノートブック v0.7（正式完了用）
2026-08-25。ChatGPT「Step0結果報告v1.1レビュー」§17の12項目を1回の実行で機械検証し，
Step 0を正式完了させる。**コミット系譜の訂正を含む**：

- **63aa853…** = 偏光予言の登録コミット（paper 3の事前登録）。Step 0パイプラインは含まない。
- **d36e756…** = A1解析（plane_mirror.py・phase2_core.py・a1_bands.csv）を凍結したコミット。
  以後これらのファイルは未変更（HEADとの差はLICENSEのみ）。**Step 0の凍結参照はこちら**。
  従来文書の「commit 63aa853の凍結pipeline」という記述は本ノートブックで訂正する。

null再現の主張水準（§11対応）：A1実行時にper-realization配列は保存されていないため，
凍結成果物は「決定論的レシピ（seed 0..999＋C_ℓファイル＋コード）」である。本実行は
(i) 全5帯域のnull median厳密一致，(ii) サンドボックス独立実行との**要素別・SHA256照合**
（クロス環境）を証拠とし，本実行の配列＋SHA256を以後の凍結成果物として保存する。

p(ρ)等は**frozen-axis conditional**（軸は同一温度データから発見——§8）。10マップは独立試行
ではなくrobustnessの証拠（§9）。

In [ ]:
import os, sys, json, hashlib, subprocess, datetime, glob
IN_COLAB = os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('Driveマウント失敗：ここで明示停止。再実行してください。')
    subprocess.run([sys.executable,'-m','pip','install','-q','healpy==1.20.0'], check=True)
    WORK='/content'; OUT='/content/drive/MyDrive/mirror_topology/runs_step0'
    PR4_DIR='/content/drive/MyDrive/phase2_null/sources'
    NULL_REF_NPZ=os.path.join(OUT,'step0_null_pm_sandbox_reference.npz')  # 任意（あれば照合）
else:
    WORK='/home/claude'; OUT='/home/claude/step0_sim/official'; PR4_DIR='/mnt/user-data/uploads'
    NULL_REF_NPZ='/home/claude/step0_null_pm.npz'
os.makedirs(OUT, exist_ok=True)
import numpy as np, healpy as hp, pandas as pd
EXPECTED = {
 "COMMIT_STEP0_FULL": "d36e7567e8a7869c0d7b84955b4139ab0e782af0",
 "COMMIT_PREDREG_FULL": "63aa853f176837a6c16dcc2851fdec6fd783b6d6",
 "SRC": {
  "src/plane_mirror.py": "1af60a031ffd6854f0c7941ce83ca2c94e9c5f3a146996151bb1f53dfb7f1fbd",
  "src/phase2_core.py": "976534f5cf6c93521a1e974dd6e175e613bc639b4b173a5cc25f8f0545963d7d",
  "data/a1_bands.csv": "14a7e7c36dc245cf8ebc654029e046f0bd7d1a2c8e9f682d8b1f7a04399c2ae9"
 },
 "MAPS": {
  "PR3_Commander": "2f88c2d385e3c96a8ead4d98254e8b92ad4f460c58810697ad61132f2ff0020b",
  "PR3_NILC": "63f3b41ea5a0e934d2d425ccb7c51b12539e4fb147e75e18a92e18e003bee352",
  "PR3_SEVEM": "c395a7fcf955560a62d7b8404eb3ff19e868d6c03c0ad83dbf99571951e12ab9",
  "PR3_SMICA": "1fda86de628c43820e999ad1564960cb5350c972c7f820b4132d6c6d149a8272",
  "Nofi_70GHz": "bac5d05118d1044c07910283a31e20d712a5f55190a3258e12c6c53fc1cc2665",
  "Nofi_94GHz": "6a24e2867e4f4aefd4f2cff7e0b591912f8f4690e5cccb24cec99f95c2047266",
  "Nofi_100GHz": "b30cb4cacb104ca1e705c320f9f4b867c0a03e4130248cca54f007a57e87b867",
  "Nofi_143GHz": "ad95a5135470d3f297267d75c5323ad30aaf454d1bf0f8a1aae076142ba0c6eb",
  "PR4_Sevem": "94b6647a220590e82162786c02a10a81d6e4421cc3949e247cdd4486cf280a1e",
  "PR4_Commander": "e9dfe38bc25d7ba22294161ffa709c1fc11914a3019eb1deaf3ed0e646dae40a"
 },
 "MASK_SRC": "027ea598135c77784000f37885c36800d1255aa7e2b5a5780514907ff05ca5ee",
 "CL_FILE": "e4d06f6afba86ba7ed3af8b032a470fec4ed01d42c292813605971fff6e6436b",
 "NULL_REF": {
  "Np_sha": "961cf71068263f58bc770f4746ecff84df536a55cce207864412f6ef28f39f09",
  "Nm_sha": "63c1b4fe97ab9b56509e49633c312b8ce4d10a158db28c09fe9de3595ab9b2d9",
  "Sp_med": "391.3361551439457",
  "Sm_med": "279.48463039847104"
 },
 "CMBANOM_COMMIT": "aaf8137427d54ce4c77e59734391aca491a4a8db"
}
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
VER = dict(python=sys.version.split()[0], numpy=np.__version__, healpy=hp.__version__)
print(VER); print('OUT =', OUT)

In [ ]:
# === 凍結リポジトリ：exact commit checkout・clean tree・ファイルSHA256（§13対応）===
REPO = os.path.join(WORK, 'pem_repo')
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git','clone','https://github.com/tsujikeita/plane-excised-mirror.git', REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','-q',EXPECTED['COMMIT_STEP0_FULL']], check=True)
COMMIT = subprocess.run(['git','-C',REPO,'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
CLEAN = subprocess.run(['git','-C',REPO,'status','--porcelain','--untracked-files=no'],
                       capture_output=True,text=True).stdout.strip() == ''
SRC_OK = {f: sha(os.path.join(REPO,f)) == h for f,h in EXPECTED['SRC'].items()}
print('commit =', COMMIT, '(exact match:', COMMIT==EXPECTED['COMMIT_STEP0_FULL'], ') clean tree:', CLEAN)
print('src SHA256一致:', SRC_OK)
assert COMMIT == EXPECTED['COMMIT_STEP0_FULL'] and CLEAN and all(SRC_OK.values())
CANOM = os.path.join(WORK, 'CMBanom')
if not os.path.isdir(CANOM):
    subprocess.run(['git','clone','--depth','1','https://github.com/LauraHerold/CMBanom.git', CANOM], check=True)
CANOM_COMMIT = subprocess.run(['git','-C',CANOM,'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
MASK_SRC = os.path.join(CANOM,'data/masks/com_mask_cutoff_0.9_nside_128.fits')
CLF = os.path.join(CANOM,'data/real/COM_PowerSpect_CMB-base-plikHM-TTTEEE-lowl-lowE-lensing-minimum-theory_R3.01.txt')
MASK_OK = sha(MASK_SRC) == EXPECTED['MASK_SRC']; CL_OK = sha(CLF) == EXPECTED['CL_FILE']
print('mask hash一致:', MASK_OK, '/ C_l hash一致:', CL_OK, '/ CMBanom commit:', CANOM_COMMIT[:12])
assert MASK_OK and CL_OK
os.makedirs('pixwin_cache', exist_ok=True)
for n in ['0016','0128']:
    fp = f'pixwin_cache/pixel_window_n{n}.fits'
    if not os.path.exists(fp):
        for br in ['master','main']:
            r = subprocess.run(['curl','-sfL','-o',fp,
                f'https://raw.githubusercontent.com/healpy/healpy-data/{br}/pixel_window_functions/pixel_window_n{n}.fits'])
            if r.returncode == 0: break
sys.path.insert(0, os.path.join(REPO,'src'))
os.chdir(WORK)   # phase2_core内の相対パス（CMBanom/…）解決のため
import phase2_core as p2, plane_mirror as pm
print('frozen modules imported from', os.path.join(REPO,'src'))

In [ ]:
# === 入力（10マップhash照合 §17-5）と統計オブジェクト構築 ===
LMAX = 128
cm = {'PR3_Commander':'commander','PR3_NILC':'nilc','PR3_SEVEM':'sevem','PR3_SMICA':'smica',
      'Nofi_70GHz':'cleaned_70GHz_v9','Nofi_94GHz':'cleaned_94GHz_v9',
      'Nofi_100GHz':'cleaned_100GHz_v9','Nofi_143GHz':'cleaned_143GHz_v9'}
PATHS = {k: os.path.join(CANOM, f'data/real/map_{v}_nside_128.fits') for k,v in cm.items()}
PATHS['PR4_Sevem'] = os.path.join(PR4_DIR,'npipe_sevem_128.fits')
PATHS['PR4_Commander'] = os.path.join(PR4_DIR,'npipe_commander_128.fits')
MAP_HASH_OK = {}
for k,pth in PATHS.items():
    assert os.path.exists(pth), f'マップ欠落: {k} -> {pth}'
    MAP_HASH_OK[k] = (sha(pth) == EXPECTED['MAPS'][k])
print('10マップhash一致:', all(MAP_HASH_OK.values()), MAP_HASH_OK if not all(MAP_HASH_OK.values()) else '')
assert all(MAP_HASH_OK.values())
T_SRC = hp.gauss_beam(np.radians(1.0), lmax=LMAX) * p2.pixwin_pad(128, LMAX)
ms = pm.with_mask(p2.MirrorStat(16, p2.make_mask(16,'full')), p2.make_mask(16,'common'))
fa = pm.FixedAxisMirror(ms, 1134)
PROC_MASK_SHA = hashlib.sha256(fa.mask.astype(np.uint8).tobytes()).hexdigest()
fl = p2.transfer(16,'planck',LMAX)
ref = pd.read_csv(os.path.join(REPO,'data/a1_bands.csv'))
DATA_ALM = {k: hp.almxfl(hp.map2alm(hp.read_map(p), lmax=LMAX), 1.0/np.maximum(T_SRC,1e-12))
            for k,p in PATHS.items()}
print(f'fsky={fa.mask.mean():.4f} processed-mask sha={PROC_MASK_SHA[:16]}… axis=pix1134(N16 RING)')

In [ ]:
# === Gate A：per-map旧S⁺再現（10マップ×5帯域・凍結statistic実使用・表を保存 §12/§17-6）===
BANDS = [(2,4),(5,8),(9,16),(17,32),(33,64)]
FROZEN_STAT_CALLS = 0
grows = []
for name, alm in DATA_ALM.items():
    r = fa.band_decompose(alm.copy(), 16, fl, BANDS); FROZEN_STAT_CALLS += 1
    for a,b in BANDS:
        key=f'S{a}_{b}'
        old = float(ref[(ref['map']==name)&(ref['band']==key)]['S_data'].iloc[0])
        rel = abs(r[key]-old)/old
        tol = 1e-12 if (a,b)!=(33,64) else 1e-4
        grows.append(dict(map=name, band=key, Splus_old=old, Splus_new=r[key],
                          abs_diff=r[key]-old, rel_diff=rel, tol=tol, PASS=bool(rel<tol)))
gate = pd.DataFrame(grows)
gate.to_csv(os.path.join(OUT,'step0_gateA_permap_v0_7.csv'), index=False)
GATEA = bool(gate['PASS'].all())
w24 = gate[gate.band=='S2_4']['rel_diff'].max()
print(f'Gate A: {"PASS" if GATEA else "FAIL"}  S2_4最大rel={w24:.2e}  '
      f'(S33_64最大rel={gate[gate.band=="S33_64"]["rel_diff"].max():.2e}・既知numpy差)')
assert GATEA

In [ ]:
# === S±拡張関数＋回帰（凍結S⁺との同一性 §17-3）===
def band_pm(alm128, bands, mondip=True):
    L = hp.Alm.getlmax(len(alm128)); ell = np.arange(L+1)
    out = {}
    for (l0,l1) in bands:
        w = ((ell>=l0)&(ell<=l1)).astype(float)*fl
        mb = hp.alm2map(hp.almxfl(alm128.copy(), w), 16)
        T = np.where(fa.mask, mb, 0.0)
        if mondip:
            T = np.where(fa.mask, np.asarray(hp.remove_dipole(hp.ma(np.where(fa.mask, mb, hp.UNSEEN)))), 0.0)
        sp = 0.5*(T+T[fa.r]); sm = 0.5*(T-T[fa.r])
        out[f'Sp{l0}_{l1}'] = float(np.sum(fa.v*sp*sp)/fa.cnt)
        out[f'Sm{l0}_{l1}'] = float(np.sum(fa.v*sm*sm)/fa.cnt)
        out[f'_T{l0}_{l1}'] = T; out[f'_mb{l0}_{l1}'] = mb
    return out
worst = 0.0
for name, alm in DATA_ALM.items():
    a = band_pm(alm.copy(), [(2,4)])
    b = fa.band_decompose(alm.copy(), 16, fl, [(2,4)])
    worst = max(worst, abs(a['Sp2_4']-b['S2_4'])/b['S2_4'])
print(f'回帰: band_pm S⁺ vs 凍結S⁺ max rel = {worst:.2e}'); assert worst < 1e-14

In [ ]:
# === Gate B：null再生成（seed 0..999）・median厳密一致・クロス環境要素別照合（§11/§17-7,8）===
import time; t0=time.time()
NB_ = {f'{t}{a}_{b}': [] for t in ('Sp','Sm') for a,b in BANDS}
for s in range(1000):
    np.random.seed(s)
    r = band_pm(hp.synalm(p2.load_fid_cl(), lmax=LMAX), BANDS)
    for a,b in BANDS:
        NB_[f'Sp{a}_{b}'].append(r[f'Sp{a}_{b}']); NB_[f'Sm{a}_{b}'].append(r[f'Sm{a}_{b}'])
NULL = {k: np.array(v) for k,v in NB_.items()}
print(f'null 1000実現 再生成 ({time.time()-t0:.0f}s)')
GATEB = True
for a,b in BANDS:
    mn = float(np.median(NULL[f'Sp{a}_{b}']))
    mo = float(ref[ref['band']==f'S{a}_{b}']['S_null_med'].iloc[0])
    rel = abs(mn-mo)/mo; ok = rel < (1e-12 if (a,b)!=(33,64) else 1e-4)
    GATEB &= ok
    print(f'  S{a}_{b}: null median new={mn:.10f} old={mo:.10f} rel={rel:.2e} {"OK" if ok else "FAIL"}')
assert GATEB
CROSS_ENV = None
if os.path.exists(NULL_REF_NPZ):
    zref = np.load(NULL_REF_NPZ)
    d1 = float(np.max(np.abs(NULL['Sp2_4'] - zref['Np']))); d2 = float(np.max(np.abs(NULL['Sm2_4'] - zref['Nm'])))
    h_match = (hashlib.sha256(NULL['Sp2_4'].tobytes()).hexdigest() == EXPECTED['NULL_REF']['Np_sha']
               and hashlib.sha256(NULL['Sm2_4'].tobytes()).hexdigest() == EXPECTED['NULL_REF']['Nm_sha'])
    CROSS_ENV = dict(max_abs_diff_Sp=d1, max_abs_diff_Sm=d2, sha256_bitlevel_match=bool(h_match))
    print(f'クロス環境照合（sandbox独立実行との要素別比較）: max|ΔS⁺|={d1:.3e} max|ΔS⁻|={d2:.3e} '
          f'SHA256一致={h_match} → {"bit-level一致" if h_match else ("要素別一致(丸め水準)" if max(d1,d2)<1e-9 else "要精査")}')
else:
    print('（参照npz未配置：クロス環境照合スキップ。SHA256は下で保存され以後の凍結成果物になる）')
np.savez(os.path.join(OUT,'step0_null_arrays_v0_7.npz'), **{k:v for k,v in NULL.items()})
NULL_SHAS = {k: hashlib.sha256(v.tobytes()).hexdigest() for k,v in NULL.items()}

In [ ]:
# === 結果：10マップ確定表（S⁻分位数・両側p・診断変種つき §17-9,10,11）===
try:
    from scipy.special import sph_harm_y
    def Ylm(l,m,th,ph): return sph_harm_y(l,m,th,ph)
except ImportError:
    from scipy.special import sph_harm
    def Ylm(l,m,th,ph): return sph_harm(m,l,ph,th)
V16 = np.array(hp.pix2vec(16, np.arange(hp.nside2npix(16)))).T
dvec = np.array(hp.pix2vec(16, 1134))
RV = V16 - 2.0*(V16@dvec)[:,None]*dvec[None,:]
thR, phR = hp.vec2ang(RV)
Np_, Nm_ = NULL['Sp2_4'], NULL['Sm2_4']; NA_ = Np_-Nm_; Nr_ = NA_/(Np_+Nm_)
def pfl(k,n=1000): return f'<= {1/n:.1e}' if k==0 else f'{max(k,1)/n:.3f}'
rows=[]
for name, alm in DATA_ALM.items():
    r = band_pm(alm.copy(), [(2,4)])
    Sp, Sm, T, mb = r['Sp2_4'], r['Sm2_4'], r['_T2_4'], r['_mb2_4']
    A = Sp-Sm; rho = A/(Sp+Sm)
    # 診断変種（凍結statisticではない・定義をラベルで明示）
    Ti = hp.get_interp_val(T, thR, phR)
    v = fa.v; cnt = fa.cnt
    Sp_i = float(np.sum(v*(0.5*(T+Ti))**2)/cnt); Sm_i = float(np.sum(v*(0.5*(T-Ti))**2)/cnt)
    ell = np.arange(LMAX+1); w24 = ((ell>=2)&(ell<=4)).astype(float)*fl
    ab = hp.almxfl(alm.copy(), w24)
    Te = np.zeros(len(thR))
    for l in range(2,5):
        for m in range(0,l+1):
            aval = ab[hp.Alm.getidx(LMAX,l,m)]
            Te += (1 if m==0 else 2)*np.real(aval*Ylm(l,m,thR,phR))
    Sp_e = float(np.sum(v*(0.5*(mb+Te))**2)/cnt); Sm_e = float(np.sum(v*(0.5*(mb-Te))**2)/cnt)
    kS=int((Np_<=Sp).sum()); kA=int((NA_<=A).sum()); kR=int((Nr_<=rho).sum())
    kSm=int((Nm_<=Sm).sum()); p2s = 2*min(kSm, 1000-kSm)/1000
    rows.append(dict(map=name, Splus=Sp, Sminus=Sm, A=A, rho=rho,
        Splus_over_null_med=Sp/np.median(Np_), Sminus_over_null_med=Sm/np.median(Nm_),
        Sminus_percentile=kSm/10.0, Sminus_p_twosided=p2s,
        k_Splus=kS, p_Splus=pfl(kS), k_A=kA, p_A=pfl(kA), k_rho=kR, p_rho_frozen_axis_conditional=pfl(kR),
        rho_interp_onT=(Sp_i-Sm_i)/(Sp_i+Sm_i), rho_exact_premondip=(Sp_e-Sm_e)/(Sp_e+Sm_e)))
df = pd.DataFrame(rows)
print(df[['map','Splus','Sminus','A','rho','Splus_over_null_med','Sminus_over_null_med',
          'Sminus_percentile','p_Splus','p_A','p_rho_frozen_axis_conditional']].to_string(
          float_format=lambda x: f'{x:.4g}'))
print(f"\nnull S⁻範囲: [q16,q84]=[{np.percentile(Nm_,16):.1f},{np.percentile(Nm_,84):.1f}] "
      f"[q2.5,q97.5]=[{np.percentile(Nm_,2.5):.1f},{np.percentile(Nm_,97.5):.1f}]  "
      f"観測S⁻分位: {df.Sminus_percentile.min():.0f}–{df.Sminus_percentile.max():.0f}%（中央域=正常）")
print(f"診断変種の一致度: max|ρ_interp−ρ|={np.max(np.abs(df.rho_interp_onT-df.rho)):.4f} "
      f"max|ρ_exact−ρ|={np.max(np.abs(df.rho_exact_premondip-df.rho)):.4f}")

In [ ]:
# === OFFICIAL判定と保存（provenance JSON §17-12）===
CONDS = dict(commit_exact=(COMMIT==EXPECTED['COMMIT_STEP0_FULL']), clean_tree=CLEAN,
             src_sha_match=all(SRC_OK.values()), frozen_stat_used=(FROZEN_STAT_CALLS>=10),
             mask_source_sha=MASK_OK, cl_file_sha=CL_OK, input_maps_sha=all(MAP_HASH_OK.values()),
             gateA_permap=GATEA, gateB_null_median=GATEB)
OFFICIAL = all(CONDS.values())
print('OFFICIAL =', OFFICIAL, CONDS)
tag = '' if OFFICIAL else '_SANITY_ONLY'
df.to_csv(os.path.join(OUT, f'step0_official_v0_7{tag}.csv'), index=False)
prov = dict(notebook='Step0 v0.7 official', timestamp=str(datetime.datetime.now()), versions=VER,
    commit_step0=COMMIT, commit_predreg=EXPECTED['COMMIT_PREDREG_FULL'],
    commit_note='63aa853…=偏光予言登録（pipeline外）／d36e756…=Step0パイプライン凍結（本実行）',
    cmbanom_commit=CANOM_COMMIT, src_sha256=EXPECTED['SRC'], map_sha256=EXPECTED['MAPS'],
    mask_source_sha256=EXPECTED['MASK_SRC'], processed_mask_sha256=PROC_MASK_SHA,
    cl_file_sha256=EXPECTED['CL_FILE'], axis=dict(nside=16, pix=1134, ordering='ring'),
    config='N16_Splanck_Kcommon_mdON_harm', fsky=float(fa.mask.mean()),
    null=dict(recipe='seed 0..999, synalm lmax128, PR3 best-fit Cl', arrays_sha256=NULL_SHAS,
              cross_env_check=CROSS_ENV,
              claim_level='per-realization regeneration; exact median agreement (5 bands); cross-env element check'),
    frozen_stat_calls=FROZEN_STAT_CALLS, conds=CONDS, official=OFFICIAL,
    p_label='p(rho) etc. are frozen-axis conditional; 10 maps are robustness, not independent trials')
json.dump(prov, open(os.path.join(OUT, f'step0_provenance_v0_7{tag}.json'),'w'), indent=2, ensure_ascii=False)
print('saved:', os.path.join(OUT, f'step0_official_v0_7{tag}.csv'), '+ provenance + gateA + null npz')
if OFFICIAL:
    print('\n=== Step 0 正式完了条件（機械検証部分）: 全通過 ===')

## 実行後にお願いすること
1. `runs_step0/` の `step0_official_v0_7.csv`・`step0_provenance_v0_7.json`・
   `step0_gateA_permap_v0_7.csv`・`step0_null_arrays_v0_7.npz` を確認・保管（Driveに残ります）。
2. 最終セルの `OFFICIAL = True` と各ゲート出力（コミット・hash・Gate A/B・クロス環境照合）を
   ChatGPTへの報告に添付。
3. 事前に `step0_null_pm_sandbox_reference.npz`（Claude納品物）を `runs_step0/` に
   置いておくと，§11のクロス環境要素別照合が自動実行されます（無くても実行は完了します）。
